In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

path = "/content/drive/MyDrive/final_year_project/maize"

os.listdir(path)

['healthy1.zip']

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/final_year_project/maize/healthy1.zip"

extract_path = "/content/tanzania_healthy"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
from pathlib import Path

image_extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

images = [
    p for p in Path(extract_path).rglob("*")
    if p.suffix in image_extensions
]

print("Total images:", len(images))

Total images: 11000


In [6]:
from PIL import Image

bad_images = []

for img_path in images:
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        bad_images.append(img_path)

print("Corrupted images:", len(bad_images))

Corrupted images: 0


In [7]:
import hashlib
from collections import defaultdict

hash_to_files = defaultdict(list)

for img_path in images:

    sha = hashlib.sha256()

    with open(img_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)

    hash_to_files[sha.hexdigest()].append(img_path)

duplicate_groups = {
    h: files
    for h, files in hash_to_files.items()
    if len(files) > 1
}

duplicate_count = sum(
    len(files) - 1
    for files in duplicate_groups.values()
)

print("Unique images:", len(hash_to_files))
print("Duplicate groups:", len(duplicate_groups))
print("Duplicate files that can be removed:", duplicate_count)

Unique images: 9647
Duplicate groups: 1196
Duplicate files that can be removed: 1353


In [9]:
from pathlib import Path
import shutil
import hashlib
from collections import defaultdict

# Paths
source_path = Path("/content/tanzania_healthy")
clean_path = Path("/content/tanzania_clean/Healthy")
duplicate_path = Path("/content/tanzania_duplicates/Healthy")

clean_path.mkdir(parents=True, exist_ok=True)
duplicate_path.mkdir(parents=True, exist_ok=True)

# Find images
image_extensions = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

images = [
    p for p in source_path.rglob("*")
    if p.is_file() and p.suffix in image_extensions
]

print("Total images found:", len(images))

Total images found: 11000


In [10]:
hash_to_files = defaultdict(list)

for img_path in images:

    sha = hashlib.sha256()

    with open(img_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)

    hash_to_files[sha.hexdigest()].append(img_path)

print("Unique images:", len(hash_to_files))

Unique images: 9647


In [11]:
unique_count = 0
duplicate_count = 0

for file_list in hash_to_files.values():

    # First copy = keep
    original = file_list[0]

    destination = clean_path / original.name

    # Prevent filename collision
    if destination.exists():
        destination = clean_path / f"{unique_count}_{original.name}"

    shutil.copy2(original, destination)
    unique_count += 1

    # Remaining copies = duplicates
    for duplicate in file_list[1:]:

        destination = duplicate_path / duplicate.name

        # Prevent filename collision
        if destination.exists():
            destination = duplicate_path / f"{duplicate_count}_{duplicate.name}"

        shutil.copy2(duplicate, destination)
        duplicate_count += 1

print("Unique images copied:", unique_count)
print("Duplicate images copied:", duplicate_count)

Unique images copied: 9647
Duplicate images copied: 1353


In [12]:
clean_images = [
    p for p in clean_path.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

duplicate_images = [
    p for p in duplicate_path.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Clean folder:", len(clean_images))
print("Duplicate folder:", len(duplicate_images))
print("Total:", len(clean_images) + len(duplicate_images))

Clean folder: 9647
Duplicate folder: 1353
Total: 11000


In [13]:
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.7 MB/s eta 0:00:00


In [14]:
from pathlib import Path
from PIL import Image
import imagehash
from tqdm.auto import tqdm

clean_path = Path("/content/tanzania_clean/Healthy")

clean_images = [
    p for p in clean_path.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Images to analyze:", len(clean_images))

phash_dict = {}

for img_path in tqdm(clean_images):
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            phash_dict[img_path] = imagehash.phash(img)
    except Exception as e:
        print("Error:", img_path, e)

print("pHashes generated:", len(phash_dict))

Images to analyze: 9647


  0%|          | 0/9647 [00:00<?, ?it/s]

pHashes generated: 9647


In [15]:
import numpy as np

paths = list(phash_dict.keys())

hashes = np.array(
    [int(str(phash_dict[p]), 16) for p in paths],
    dtype=np.uint64
)

print("Number of hashes:", len(hashes))

Number of hashes: 9647


In [16]:
from collections import Counter

distance_counts = Counter()

# We only calculate the upper triangle:
# each pair is checked once.

for i in tqdm(range(len(hashes))):

    xor_values = np.bitwise_xor(hashes[i], hashes[i+1:])

    # Count number of 1 bits = Hamming distance
    distances = np.array(
        [x.bit_count() for x in xor_values]
    )

    # Keep only distances <= 24
    close_distances = distances[distances <= 24]

    distance_counts.update(close_distances.tolist())

print("Done.")

  0%|          | 0/9647 [00:00<?, ?it/s]

Done.


In [17]:
print("\nHamming distance distribution:\n")

for d in range(25):
    print(f"Distance {d:2d}: {distance_counts[d]}")


Hamming distance distribution:

Distance  0: 0
Distance  1: 0
Distance  2: 0
Distance  3: 0
Distance  4: 0
Distance  5: 0
Distance  6: 1
Distance  7: 0
Distance  8: 5
Distance  9: 0
Distance 10: 18
Distance 11: 0
Distance 12: 229
Distance 13: 0
Distance 14: 1890
Distance 15: 0
Distance 16: 12430
Distance 17: 0
Distance 18: 64075
Distance 19: 0
Distance 20: 260566
Distance 21: 0
Distance 22: 842544
Distance 23: 0
Distance 24: 2148890


In [18]:
augmentation_keywords = [
    "aug",
    "augment",
    "flip",
    "flipped",
    "rotate",
    "rotation",
    "zoom",
    "crop",
    "brightness",
    "contrast",
    "noise",
    "shear"
]

augmented_name_matches = []

for img_path in clean_images:

    name = img_path.name.lower()

    if any(keyword in name for keyword in augmentation_keywords):
        augmented_name_matches.append(img_path)

print(
    "Images whose filenames suggest augmentation:",
    len(augmented_name_matches)
)

for img in augmented_name_matches[:30]:
    print(img.name)

Images whose filenames suggest augmentation: 0


In [19]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import numpy as np
from tqdm import tqdm

# --------------------------------------------------
# PATHS
# --------------------------------------------------

SOURCE_DIR = Path("/content/tanzania_clean/Healthy")

FEATURE_DIR = Path("/content/healthy_features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# DEVICE
# --------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# --------------------------------------------------
# IMAGE TRANSFORMATION
# --------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --------------------------------------------------
# DATASET
# --------------------------------------------------

class HealthyDataset(Dataset):

    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        try:
            image = Image.open(path).convert("RGB")
            image = transform(image)
            return image, str(path)

        except Exception:
            return None, str(path)


image_paths = sorted([
    p for p in SOURCE_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print("Images found:", len(image_paths))

# --------------------------------------------------
# RESNET50 FEATURE EXTRACTOR
# --------------------------------------------------

weights = torchvision.models.ResNet50_Weights.DEFAULT

model = torchvision.models.resnet50(weights=weights)

# Remove final classification layer
model.fc = torch.nn.Identity()

model = model.to(device)
model.eval()

print("ResNet feature extractor ready.")

Using device: cuda
Images found: 9647
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s]


ResNet feature extractor ready.


In [20]:
dataset = HealthyDataset(image_paths)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

all_features = []
all_paths = []

with torch.no_grad():

    for images, paths in tqdm(loader):

        valid_indices = [
            i for i, img in enumerate(images)
            if img is not None
        ]

        if len(valid_indices) == 0:
            continue

        images = images[valid_indices].to(device)

        features = model(images)

        # Normalize feature vectors
        features = torch.nn.functional.normalize(
            features,
            p=2,
            dim=1
        )

        all_features.append(
            features.cpu().numpy()
        )

        all_paths.extend(
            [paths[i] for i in valid_indices]
        )

features = np.concatenate(all_features, axis=0)

print("Feature shape:", features.shape)
print("Images represented:", len(all_paths))

100%|██████████| 151/151 [01:43<00:00,  1.46it/s]

Feature shape: (9647, 2048)
Images represented: 9647


In [21]:
from sklearn.cluster import MiniBatchKMeans

TARGET_IMAGES = 2500

kmeans = MiniBatchKMeans(
    n_clusters=TARGET_IMAGES,
    random_state=42,
    batch_size=256,
    n_init=3
)

cluster_labels = kmeans.fit_predict(features)

print("Clustering completed.")
print("Clusters:", len(np.unique(cluster_labels)))

Clustering completed.
Clusters: 2378


In [22]:
selected_indices = []

for cluster_id in range(kmeans.n_clusters):

    indices = np.where(cluster_labels == cluster_id)[0]

    if len(indices) == 0:
        continue

    cluster_features = features[indices]

    center = kmeans.cluster_centers_[cluster_id]

    distances = np.linalg.norm(
        cluster_features - center,
        axis=1
    )

    best_index = indices[np.argmin(distances)]

    selected_indices.append(best_index)

print("Selected images:", len(selected_indices))

Selected images: 2378


In [23]:
from pathlib import Path
import shutil
from tqdm import tqdm

# Final selected images in Colab
SOURCE_DIR = Path("/content/tanzania_clean/Healthy")

# Google Drive destination
DRIVE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/maize/Selected_Healthy_tan"
)

DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy selected images
for idx in tqdm(selected_indices):

    source = Path(all_paths[idx])
    destination = DRIVE_DIR / source.name

    shutil.copy2(source, destination)

print("Copy completed!")

100%|██████████| 2378/2378 [00:30<00:00, 78.77it/s]

Copy completed!
